In [ ]:
#Building epochs for Onset-locked and Offset-Locked with ICA filtering
from pathlib import Path
import gc
import re
import mne
import numpy as np
import pandas as pd
from mne.preprocessing import ICA

mne.use_log_level("warning")

INPUT_DIR  = Path("/content/drive/...")

ONSET_DIR  = Path("/content/drive/MyDrive/...")
OFFSET_DIR = Path("/content/drive/MyDrive/...")
ONSET_DIR.mkdir(parents=True, exist_ok=True)
OFFSET_DIR.mkdir(parents=True, exist_ok=True)

ONSET_OPT_DICT = {50: 'Raw', 51: 'AM', 52: 'FM', 56: 'Binaural'}
ONSET_TGT_DICT = {70: 'Raw', 71: 'AM', 72: 'FM', 76: 'Binaural'}

RESAMPLE_ICA = 100


def get_trigger(desc):
    m = re.search(r"Stimulus/S\s*(\d+)", desc)
    return int(m.group(1)) if m else None


def build_metadata(events):
    in_trial = False
    trial_events = []
    on_evts, on_meta = [], []
    off_evts, off_meta = [], []

    for ev in events:
        code = ev[2]
        if code == 101:
            in_trial = True
            trial_events = []
        elif code == 102 and in_trial:
            onsets = [e for e in trial_events if e[2] in ONSET_OPT_DICT or e[2] in ONSET_TGT_DICT]
            offsets = [e for e in trial_events if e[2] in {69, 89}]

            is_att = not any(e[2] in ONSET_TGT_DICT for e in onsets)
            trial_type = 'Attention_Check' if is_att else 'Normal'

            tpos = -1
            if not is_att:
                for idx, e in enumerate(onsets):
                    if e[2] in ONSET_TGT_DICT:
                        tpos = idx + 1
                        break

            for idx, e in enumerate(onsets):
                is_tgt = e[2] in ONSET_TGT_DICT
                tagger = ONSET_TGT_DICT[e[2]] if is_tgt else ONSET_OPT_DICT[e[2]]
                on_evts.append([e[0], 0, 2 if is_tgt else 1])
                on_meta.append({
                    'Trial_Type': trial_type,
                    'Word_Position': idx + 1,
                    'Is_Target': is_tgt,
                    'Target_Position': tpos,
                    'Tagger_Type': tagger,
                    'Original_Code': e[2],
                })

            for idx, e in enumerate(offsets):
                is_tgt = (e[2] == 89)
                tagger = 'Unknown'
                if idx < len(onsets):
                    oc = onsets[idx][2]
                    tagger = ONSET_TGT_DICT.get(oc, ONSET_OPT_DICT.get(oc, 'Unknown'))
                off_evts.append([e[0], 0, 2 if is_tgt else 1])
                off_meta.append({
                    'Trial_Type': trial_type,
                    'Word_Position': idx + 1,
                    'Is_Target': is_tgt,
                    'Target_Position': tpos,
                    'Tagger_Type': tagger,
                    'Original_Code': e[2],
                })

            in_trial = False
        elif in_trial:
            trial_events.append(ev)

    on_arr = np.array(on_evts) if on_evts else np.array([])
    on_df = pd.DataFrame(on_meta) if on_meta else pd.DataFrame()
    off_arr = np.array(off_evts) if off_evts else np.array([])
    off_df = pd.DataFrame(off_meta) if off_meta else pd.DataFrame()
    return on_arr, on_df, off_arr, off_df


def process_subject(raw_path):
    subj = re.search(r"(sub-\d+)", raw_path.name).group(1)
    print(f"\n{'='*40}\nProcessing {subj}...")

    raw = None
    raw_ica_fit = None
    ica = None
    epochs = None

    try:
        # resample
        raw = mne.io.read_raw_fif(raw_path, preload=True, verbose=False)
        raw.resample(RESAMPLE_ICA, verbose=False)
        gc.collect()

        raw_ica_fit = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)
        print("  Fitting ICA...")
        ica = ICA(n_components=15, method="fastica", random_state=42, max_iter="auto")
        ica.fit(raw_ica_fit)
        del raw_ica_fit
        raw_ica_fit = None
        gc.collect()

        # blink component and removing
        try:
            eog_idx, _ = ica.find_bads_eog(raw, ch_name="Fp1", verbose=False)
        except Exception as e:
            print(f"  blink detection failed: {e}")
            eog_idx = []
        print(f"  Removing blink components: {eog_idx}")

        ica.apply(raw, exclude=eog_idx, verbose=False)
        del ica
        ica = None
        gc.collect()

        events, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)
        on_arr, on_df, off_arr, off_df = build_metadata(events)

        # ONSET
        if len(on_arr) > 0:
            print("  Extracting ONSET...")
            epochs = mne.Epochs(
                raw, on_arr,
                event_id={"option": 1, "target": 2},
                tmin=-0.2, tmax=1.0,
                baseline=(-0.2, 0.0),
                metadata=on_df,
                picks="eeg",
                event_repeated="drop",
                preload=True, verbose=False,
            )
            epochs.save(ONSET_DIR / f"{subj}-onset-epo.fif", overwrite=True)
            print(f"     Saved onset — {len(epochs)} epochs")
            del epochs
            epochs = None
            gc.collect()

        #OFFSET
        if len(off_arr) > 0:
            print("  Extracting OFFSET...")
            epochs = mne.Epochs(
                raw, off_arr,
                event_id={"option": 1, "target": 2},
                tmin=-0.5, tmax=0.6,
                baseline=None,
                metadata=off_df,
                picks="eeg",
                event_repeated="drop",
                preload=True, verbose=False,
            )
            epochs.save(OFFSET_DIR / f"{subj}-offset-epo.fif", overwrite=True)
            print(f"     Saved offset — {len(epochs)} epochs")
            del epochs
            epochs = None
            gc.collect()

    except Exception as e:
        print(f"  Error on {subj}: {e}")
        import traceback
        traceback.print_exc()
    finally:
        for obj in [raw, raw_ica_fit, ica, epochs]:
            if obj is not None:
                del obj
        gc.collect()


raw_files = sorted(
    f for f in INPUT_DIR.glob("sub-*_ses-01-raw.fif")
    if "sub-03" not in f.name and "_2" not in f.name and "sub-michal" not in f.name
)
print(f"Found {len(raw_files)} subjects.")

for raw_path in raw_files:
    process_subject(raw_path)

print("\nALL ICA EPOCHS DONE!")